# 실습2 · 코퍼스와 형태소 분석

검색 대상(코퍼스)이 어떻게 생겼는지 보고, 키워드 검색의 단위인 **토큰**을 PostgreSQL 이 어떻게 만드는지 확인한다.
강의 : 「코퍼스 — 검색의 토대」 · 「키워드 검색과 토큰」 · 「형태소 분석기와 토큰화」

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 라이브러리와 DB 연결 |
| 3 | 코퍼스 구성 — 출처 · 분야 · 질문 수 |
| 4 | 지문 길이 분포 |
| 5 | mecab-ko 원래 출력 — 품사 태그 |
| 6 | simple vs korean — 조사가 붙은 토큰 |
| 7 | ts_debug — 토큰마다 어떤 사전이 처리했나 |
| 8 | tsquery 세 가지와 매칭 수 |
| 9 | GIN 인덱스를 타는지 |
| 10 | 문장형 질문의 함정 — AND 는 거의 못 찾음 |

실습1(데이터 적재)을 마친 뒤 실행. 임베딩은 필요 없다. **Run → Run All Cells**

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"

TEXT     = "고양이가 사료를 먹었습니다. 고양이 사료 추천"          # 셀 6 에서 토큰화할 문장
QUESTION = "말라카이트에서 나온 색깔을 사용한 에디션은?"           # 셀 10 의 문장형 질문 (정답 지문 id 1)

### 셀 2 · 라이브러리와 DB 연결
`sql(쿼리)` 는 결과를 표로 보여 주는 도구. 이후 모든 셀이 쓴다.

In [ ]:
# 셀 2 · 라이브러리와 DB 연결
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DB_URL, autocommit=True)
conn.prepare_threshold = None      # SET 으로 바꾼 설정이 매번 반영되게 (실행 계획 재사용 끔)
register_vector(conn)              # numpy 배열 ↔ vector 타입


def sql(query, params=None):
    """SELECT 결과를 표(DataFrame)로"""
    cur = conn.execute(query, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]) if cur.description else None

print(conn.execute("SELECT count(*) FROM docs").fetchone()[0], "지문 /",
      conn.execute("SELECT count(*) FROM questions").fetchone()[0], "질문")

### 셀 3 · 코퍼스 구성
검색 품질은 모델보다 코퍼스가 먼저 정한다. 무엇이 몇 개 들어 있는지부터 센다.
- 출처 3곳 : 위키백과 · 아크로팬 · 한국경제. 위키 지문은 분야(category)가 없다
- 질문수 : 그 출처의 지문에 딸린 질문. 지문 하나에 질문이 1~여러 개

In [ ]:
# 셀 3 · 코퍼스 구성 — 출처 · 분야 · 질문 수
sql("""
SELECT d.source                          AS 출처,
       count(DISTINCT d.id)              AS 지문수,
       count(DISTINCT d.category)        AS 분야수,
       count(q.id)                       AS 질문수,
       round(avg(length(d.content)))     AS 평균글자,
       round(avg(d.doclen))              AS 평균형태소
FROM docs d LEFT JOIN questions q ON q.doc_id = d.id
GROUP BY d.source ORDER BY 지문수 DESC
""")

### 셀 4 · 지문 길이 분포
지문 하나를 그대로 검색 단위(청크 1개)로 쓴다. 길이가 고르면 BM25 의 길이 보간이 할 일이 적다.
- 왼쪽 : 글자 수 · 가운데 : 형태소 수(doclen) · 오른쪽 : 둘의 관계

In [ ]:
# 셀 4 · 지문 길이 분포
L = sql("SELECT length(content) AS 글자, doclen AS 형태소, source FROM docs")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(L["글자"], bins=50, color="#3D484A"); ax[0].set_title("지문 글자 수")
ax[1].hist(L["형태소"], bins=50, color="#FF7043"); ax[1].set_title("지문 형태소 수 (doclen)")
for s, c in zip(["wikipedia", "hankyung", "acrofan"], ["#009CA6", "#3D484A", "#FF7043"]):
    m = L["source"] == s
    ax[2].scatter(L.loc[m, "글자"], L.loc[m, "형태소"], s=3, alpha=0.4, color=c, label=s)
ax[2].set_xlabel("글자"); ax[2].set_ylabel("형태소"); ax[2].legend(markerscale=4)
plt.tight_layout(); plt.show()
L[["글자", "형태소"]].describe().round(0).T

### 셀 5 · mecab-ko 원래 출력 — 품사 태그
`mecabko_analyze()` 는 textsearch_ko 가 주는 함수. 형태소마다 품사(type)가 붙는다.
- NNG 일반명사 · JKS 주격조사 · JKB 부사격조사 · VV 동사 · EP 선어말어미 · EC·EF 어미

In [ ]:
# 셀 5 · mecab-ko 원래 출력
for s in ["아버지가방에들어가신다", "가셨습니다", "인공지능모델학습데이터"]:
    print(s)
    display(sql("SELECT word AS 형태소, type AS 품사 FROM mecabko_analyze(%s)", (s,)).T)

### 셀 6 · simple vs korean
- `simple` : 공백·기호로만 자름 → '고양이가' 와 '고양이' 가 다른 토큰
- `korean` : 형태소 분석 → 조사 · 어미를 떼고 용언은 어간으로. 품사 태그는 버리고 형태소만 남김
- 마지막 표 : 조사가 붙은 첫 문장만으로 '고양이' 검색이 되는가

In [ ]:
# 셀 6 · simple vs korean
display(sql("""SELECT cfg AS 설정, to_tsvector(cfg::regconfig, %(t)s)::text AS tsvector
               FROM unnest(ARRAY['simple', 'korean']) cfg""", {"t": TEXT}))
first = TEXT.split(".")[0]
sql("""SELECT cfg AS 설정, to_tsvector(cfg::regconfig, %(s)s) @@ to_tsquery(cfg::regconfig, '고양이') AS 고양이로_찾힘
       FROM unnest(ARRAY['simple', 'korean']) cfg""", {"s": first})

### 셀 7 · ts_debug — 토큰마다 어떤 사전이 처리했나
영문은 `english_stem`, 숫자는 `simple`, 한글은 `korean_stem`(mecab-ko). 사전이 돌려준 것이 lexemes = tsvector 에 들어가는 형태소.

In [ ]:
# 셀 7 · ts_debug
sql("""SELECT alias AS 종류, token AS 토큰, dictionaries AS 사전, lexemes AS 결과
       FROM ts_debug('korean', 'PostgreSQL 17에서 벡터 검색을 테스트했다') WHERE alias <> 'blank'""")

### 셀 8 · tsquery 세 가지와 매칭 수
질의도 같은 형태소 분석을 거쳐 tsquery 가 된다. `tsv @@ tsquery` 가 매칭.
- `plainto_tsquery` : 모든 형태소를 AND(&)
- `websearch_to_tsquery` : 검색엔진 문법(or, -제외, "구문")
- `to_tsquery` 의 `<->` : 바로 붙어 나와야 매칭(구문 검색)

In [ ]:
# 셀 8 · tsquery 세 가지와 매칭 수
sql("""
SELECT 종류, tq::text AS tsquery, (SELECT count(*) FROM docs WHERE tsv @@ tq) AS 매칭지문
FROM (VALUES
  ('plainto  AND',     plainto_tsquery('korean', '반도체 수출')),
  ('websearch OR',     websearch_to_tsquery('korean', '반도체 or 자동차 -수출')),
  ('구문 <-> 붙어서',  to_tsquery('korean', '인공 <-> 지능')),
  ('떨어져도 됨 &',    to_tsquery('korean', '인공 & 지능'))
) v(종류, tq)
""")

### 셀 9 · GIN 인덱스를 타는지
`docs_tsv_gin` : 형태소 → 그 형태소를 가진 지문 목록(역색인). 실행 계획에 **Bitmap Index Scan on docs_tsv_gin** 이 보이면 인덱스로 찾은 것.

In [ ]:
# 셀 9 · GIN 인덱스를 타는지
for (line,) in conn.execute("""EXPLAIN (ANALYZE, COSTS OFF)
        SELECT id FROM docs WHERE tsv @@ plainto_tsquery('korean', '반도체 수출')"""):
    print(line)

### 셀 10 · 문장형 질문의 함정
질문 문장을 그대로 `plainto_tsquery` 에 넣으면 형태소 5개가 모두 & 로 묶인다 → 다섯 개를 다 가진 지문만 매칭.
& 를 | (OR) 로 바꾸면 하나라도 가진 지문이 전부 후보 → 순위를 매기는 함수(ts_rank · BM25)가 필요해진다. 다음 실습3.

In [ ]:
# 셀 10 · 문장형 질문 — AND vs OR
sql("""
WITH a AS (SELECT plainto_tsquery('korean', %(q)s) AS tq),
     o AS (SELECT replace(tq::text, '&', '|')::tsquery AS tq FROM a)
SELECT 'AND' AS 묶음, a.tq::text AS tsquery, (SELECT count(*) FROM docs WHERE tsv @@ a.tq) AS 매칭지문,
       (SELECT tsv @@ a.tq FROM docs WHERE id = 1) AS 정답지문_매칭 FROM a
UNION ALL
SELECT 'OR', o.tq::text, (SELECT count(*) FROM docs WHERE tsv @@ o.tq), (SELECT tsv @@ o.tq FROM docs WHERE id = 1) FROM o
""", {"q": QUESTION})